In [3]:
import numpy as np 
import pandas as pd

In [30]:
def generate_total_seqs_per_donor(load_path):
    '''
    All data that goes here must be before redundant seqs are removed
        cdrh3_seqs_by_donor = no seqs were removed.
    '''
    import numpy as np
    donors_list = np.load(f'{load_path}donors_list.npy')
    donors_list_unique = np.unique(donors_list)
    total_seqs_per_donor = []
    donor_name = []
    for donors in range(len(donors_list_unique)):
        donor = donors_list_unique[donors]

        # get the number of sequences from each donor in the full OAS database
        counter = []
        with open(f"/Users/ssolieva/Desktop/Kulp_lab/projects/OAS_database_searches/donor_counts/{donor}.out", "r") as fd: # import the results file
            file_contents = fd.read().splitlines() # read in the results
            for i in range(len(file_contents)):
                counter.append(int(file_contents[i]))
            #print(f'{donor}, Number of sequences in OAS database: {sum(counter)}')
            n_seqs_donor = sum(counter)
        fd.close()
        total_seqs_per_donor.append(n_seqs_donor)
        donor_name.append(donor)
    return donor_name, total_seqs_per_donor

In [37]:
def save_out_data_with_cdr_lengths(search_name, cutoff):
    load_path=f'/Users/ssolieva/Desktop/github_repo/Q23_paper/OAS_database_searches/V033/{search_name}/data/parsed_logfile/'

    donor_name, n_total_seqs = generate_total_seqs_per_donor(load_path)
    
    
    load_path=f'/Users/ssolieva/Desktop/github_repo/Q23_paper/OAS_database_searches/V033/{search_name}/data/parsed_logfile/'
    
    all_hits = np.load(f'{load_path}cdrh3_seqs_by_donor.npy', allow_pickle=True)
    all_hits_min_resis_per_donor = []
    for donor in range(len(all_hits)):
        all_hits_min_resis_one_donor = []
        for cdr in range(len(all_hits[donor])):
            if len(all_hits[donor][cdr]) >= cutoff:
                all_hits_min_resis_one_donor.append(all_hits[donor][cdr])
        all_hits_min_resis_per_donor.append(all_hits_min_resis_one_donor)
    n_all_hits = []
    for i in range(len(all_hits_min_resis_per_donor)):
        n_all_hits.append(int(len(all_hits_min_resis_per_donor[i])))
        
    collapsed_hits = np.load(f'{load_path}redundant_seqs_removed_within_donors_cdrh3_final.npy', allow_pickle=True)
    collapsed_hits_min_resis_per_donor = []
    for donor in range(len(collapsed_hits)):
        collapsed_hits_min_resis_one_donor = []
        for cdr in range(len(collapsed_hits[donor])):
            if len(collapsed_hits[donor][cdr]) >= cutoff:
                collapsed_hits_min_resis_one_donor.append(collapsed_hits[donor][cdr])
        collapsed_hits_min_resis_per_donor.append(collapsed_hits_min_resis_one_donor)
    n_collapsed_hits = []
    for i in range(len(collapsed_hits_min_resis_per_donor)):
        n_collapsed_hits.append(int(len(collapsed_hits_min_resis_per_donor[i])))
    
    n_seqs_removed = []
    for i in range(len(n_all_hits)):
        n_seqs_removed.append(int(n_all_hits[i]-n_collapsed_hits[i]))
        
    n_total_seqs_minus_n_redundant_hits = []
    for i in range(len(n_all_hits)):
        n_total_seqs_minus_n_redundant_hits.append(int(n_total_seqs[i]-n_seqs_removed[i]))
    
    
    original_frequency_estimate=[]
    for i in range(len(n_all_hits)):
        original_frequency_estimate.append(float((n_all_hits[i]/n_total_seqs[i])*1000000))
        
    conservative_frequency_estimate=[]
    for i in range(len(n_all_hits)):
        conservative_frequency_estimate.append(float((n_collapsed_hits[i]/n_total_seqs_minus_n_redundant_hits[i])*1000000))
      
        
    data = {
        "donor_name":       donor_name,
        "n_total_seqs":     n_total_seqs,
        "n_all_hits":       n_all_hits,
        "n_nonredundant_hits": n_collapsed_hits, 
        "n_redundant_hits":    n_seqs_removed, 
        "n_total_seqs_minus_n_redundant_hits" : n_total_seqs_minus_n_redundant_hits,
        "original_frequency_estimate":original_frequency_estimate,
        "conservative_frequency_estimate": conservative_frequency_estimate
    }
    
    df = pd.DataFrame(data)
    #print(df) 
    
    df.to_csv(f'V033_{search_name}_min_H3_length_{cutoff}.csv')
    
    #return n_collapsed_hits,n_all_hits

In [38]:
save_out_data_with_cdr_lengths('search1', 23)
save_out_data_with_cdr_lengths('search1', 20)

In [39]:
save_out_data_with_cdr_lengths('search2', 23)
save_out_data_with_cdr_lengths('search2', 20)

In [40]:
save_out_data_with_cdr_lengths('search3', 23)
save_out_data_with_cdr_lengths('search3', 20)

In [10]:
def save_out_data(search_name):
    #load_path=f'/Users/ssolieva/Desktop/Kulp_lab/projects/OAS_database_searches/V033/updated_searches_05222024/{search_name}'
    load_path=f'/Users/ssolieva/Desktop/github_repo/Q23_paper/OAS_database_searches/V033/{search_name}/data/parsed_logfile/'

    donor_name, n_total_seqs = generate_total_seqs_per_donor(load_path)
    
    all_hits = np.load(f'{load_path}cdrh3_seqs_by_donor.npy', allow_pickle=True)
    n_all_hits = []
    for i in range(len(all_hits)):
        n_all_hits.append(int(len(all_hits[i])))
        
    collapsed_hits = np.load(f'{load_path}redundant_seqs_removed_within_donors_cdrh3_final.npy', allow_pickle=True)
    n_collapsed_hits = []
    for i in range(len(collapsed_hits)):
        n_collapsed_hits.append(int(len(collapsed_hits[i])))
    
    n_seqs_removed = []
    for i in range(len(n_all_hits)):
        n_seqs_removed.append(int(n_all_hits[i]-n_collapsed_hits[i]))
        
    n_total_seqs_minus_n_redundant_hits = []
    for i in range(len(n_all_hits)):
        n_total_seqs_minus_n_redundant_hits.append(int(n_total_seqs[i]-n_seqs_removed[i]))
    
    
    original_frequency_estimate=[]
    for i in range(len(n_all_hits)):
        original_frequency_estimate.append(float((n_all_hits[i]/n_total_seqs[i])*1000000))
        
    conservative_frequency_estimate=[]
    for i in range(len(n_all_hits)):
        conservative_frequency_estimate.append(float((n_collapsed_hits[i]/n_total_seqs_minus_n_redundant_hits[i])*1000000))
      
        
    data = {
        "donor_name":       donor_name,
        "n_total_seqs":     n_total_seqs,
        "n_all_hits":       n_all_hits,
        "n_nonredundant_hits": n_collapsed_hits, 
        "n_redundant_hits":    n_seqs_removed, 
        "n_total_seqs_minus_n_redundant_hits" : n_total_seqs_minus_n_redundant_hits,
        "original_frequency_estimate":original_frequency_estimate,
        "conservative_frequency_estimate": conservative_frequency_estimate
    }
    
    df = pd.DataFrame(data)
    #print(df) 
    
    df.to_csv(f'V033_{search_name}.csv')

In [11]:
search_names = ['search1', 'search2', 'search3']

for search_name in search_names:
    save_out_data(search_name)